# Kaggle 免费 GPU 跑 Qwen3-Coder-30B-A3B（写代码，vLLM 双卡版）

目标：Kaggle T4 x2（每张 16GB，共 32GB）上用 vLLM 跑 `QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ`（4-bit，约 16.8GB），走 OpenAI 兼容接口给本地 VS Code / Continue 用。

为什么不用 Ollama：Ollama 默认只用一张卡，19GB 的模型单卡装不下；vLLM 用 `tensor-parallel-size=2` 把模型拆到两张 T4 上，这是这套配置的关键。

前置条件（新建 notebook 时就要选好）：
- Accelerator 选 **GPU T4 x2**；右侧 Settings 里 **Internet 必须 ON**（否则拉不到 vLLM 和模型）。
- 免费额度约 30 小时/周，单次会话最长 12 小时；右上角看会话计时，别挂着空跑。
- T4 是 Turing 架构（算力 7.5），不支持 FlashAttention 2，所以 vLLM 要加 `--enforce-eager`，不要开 CUDA graphs。
- 默认按你的主业准备：C/C++、Qt/QML；这个模型写 Python 同样适用，换提示词即可，不用换模型。
- 本 notebook 未在 Kaggle 上实测过：前面先是检查 cell（GPU、磁盘、联网），失败会直接报错停在那一步，按报错修，不会悄悄跳过。

In [ ]:
# 1. 先验环境：必须是 2 张 T4、磁盘够 25GB+、能连 Hugging Face（任一项不对就先停）
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!nvidia-smi -L
!df -h /kaggle/working | tail -1
!python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpus', torch.cuda.device_count())"
!curl -sI --max-time 15 https://huggingface.co | head -1

In [ ]:
# 2. 装 vLLM（要几分钟，输出很长，只看最后几行有没有报错）
#    Kaggle 镜像自带 torch，pip 装 vLLM 会顺带装不少依赖；装完如果提示 torch 版本冲突，先别急着修，跑第 5 步看 vLLM 能不能起来。
!pip install -q vllm 2>&1 | tail -5
!python3 -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 3. 先把模型下到 /kaggle/working（约 16.8GB，下完 vLLM 启动就不用再等下载）
#    仓库：QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ，Apache 2.0 开源权重。
#    下一次开新会话要重新下，Kaggle 会话之间不保存。
from huggingface_hub import snapshot_download
p = snapshot_download(repo_id='QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ', local_dir='/kaggle/working/Qwen3-Coder-30B-A3B-AWQ')
print('downloaded to', p)
!du -sh /kaggle/working/Qwen3-Coder-30B-A3B-AWQ
!df -h /kaggle/working | tail -1

In [ ]:
# 4. 起 vLLM OpenAI 兼容服务（后台跑，日志写到 /kaggle/working/vllm.log）
#    关键参数：--tensor-parallel-size 2（双卡拆分）、--enforce-eager（T4 必须）、
#    --max-model-len 65536（19GB 模型 + KV cache 在 32GB 里留余量；想加长上下文再改 32768，但要先看显存够不够）、
#    --gpu-memory-utilization 0.90、--max-num-seqs 2 --enable-auto-tool-choice --tool-call-parser qwen3_coder（T4 上并发别贪多）。
import os, subprocess
os.makedirs('/kaggle/working', exist_ok=True)
log = open('/kaggle/working/vllm.log', 'wb')
cmd = ['python3', '-m', 'vllm.entrypoints.openai.api_server',
       '--model', '/kaggle/working/Qwen3-Coder-30B-A3B-AWQ',
       '--served-model-name', 'qwen3-coder-30b',
       '--host', '0.0.0.0', '--port', '8000',
       '--tensor-parallel-size', '2',
       '--max-model-len', '65536',
       '--gpu-memory-utilization', '0.90',
       '--max-num-seqs', '2',
       '--enforce-eager', '--enable-auto-tool-choice', '--tool-call-parser', 'qwen3_coder', '--disable-log-stats']
print(' '.join(cmd))
subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print('vllm started, log: /kaggle/working/vllm.log')

In [ ]:
# 5. 等服务起来（加载 19GB 模型到双卡要几分钟，别急；超过 20 分钟没起来就看 vllm.log 尾部）
import time, urllib.request, json
for i in range(120):
    try:
        with urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5) as r:
            print('vllm is up:', r.read().decode()[:500])
            break
    except Exception as e:
        if i % 12 == 0:
            print(f'waiting {i*10}s ... {e}')
        time.sleep(10)
else:
    raise RuntimeError('vLLM 20 分钟没起来，看 /kaggle/working/vllm.log 尾部报错')

In [ ]:
# 6. 冒烟测试：走 OpenAI 兼容接口，一段 C++17 代码补全，确认模型真在双卡上跑、能出代码
import json, urllib.request
req = urllib.request.Request('http://localhost:8000/v1/chat/completions', data=json.dumps({
    'model': 'qwen3-coder-30b', 'temperature': 0.2, 'max_tokens': 600,
    'messages': [{'role': 'user', 'content': '用 C++17 写一个函数：输入 std::vector<int>，返回其中出现次数最多的元素，写清时间复杂度，只给代码加一行注释说明思路。'}],
}).encode(), headers={'Content-Type': 'application/json'})
resp = json.loads(urllib.request.urlopen(req, timeout=300).read())
print(resp['choices'][0]['message']['content'])
print('usage:', resp.get('usage'))

## 7. 接入本地 VS Code（可选，跑通上面再做）

Kaggle 不直接对外暴露端口，走 VS Code Tunnel（出站联网，不用开端口）：

```bash
# 在 notebook 的 terminal 里（或加一个 ! 开头的 cell）
curl -Lk 'https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64' --output /kaggle/working/vscode_cli.tar.gz
tar -xf /kaggle/working/vscode_cli.tar.gz -C /kaggle/working/
/kaggle/working/code tunnel --accept-server-license-terms
```

按提示在你自己电脑的浏览器里完成 GitHub/Microsoft 登录（设备码授权，别把码贴给别人）。
然后本地 VS Code 左下角 → “连接到隧道” → 选这个隧道；隧道里的 VS Code 装 **Continue** 插件，配置见同目录 `continue-config-qwen3-coder-30b.yaml`（OpenAI 兼容接口，指向隧道内的 `http://localhost:8000/v1`，模型名 `qwen3-coder-30b`，因为 Continue、vLLM 跑在同一个 Kaggle 会话里）。

## 注意事项

- 模型只在当前会话有效：Kaggle 会话结束，`/kaggle/working` 下的东西不保留，下次要重新下约 16.8GB + 重装 vLLM，想省时间就一次把活干完。
- 别贪大：`Qwen3-Coder-480B` 不用想，T4 x2 装不下；`Qwen2.5-Coder-32B` 也是约 20GB，同样要双卡 vLLM，做法一样、换模型路径就行。
- 速度预期：T4 上 30B-A3B 是 MoE（每 token 只激活约 3.3B），比同等 dense 30B 快，但 T4 本身不快，别跟本地 4090 比；上下文 16384 是留了余量的稳妥值。
- 如果 vLLM 装不上 / 起不来，先看 `/kaggle/working/vllm.log` 尾部，再看第 2 步 pip 有没有 torch 冲突；Ollama 版 `qwen3-coder:30b` 是备选，但 Ollama 默认单卡，19GB 在单张 16GB T4 上要靠 offload，速度和稳定性都不如 vLLM 双卡。